In [1]:
!pip install spacy scikit-learn nltk pandas matplotlib seaborn wordcloud plotly
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 92.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys

project_path = '/content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM'

sys.path.append(project_path)

In [4]:
import pandas as pd

df = pd.read_csv(f'{project_path}/data/BBC News Train.csv')
df.head()

,ArticleId,Text,Category
0,1833,worldcom ex-boss launches defence lawyers defe...,business
1,154,german business confidence slides german busin...,business
2,1101,bbc poll indicates economic gloom citizens in ...,business
3,1976,lifestyle governs mobile choice faster bett...,tech
4,917,enron bosses in $168m payout eighteen former e...,business


In [5]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1490 entries, 0 to 1489
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ArticleId  1490 non-null   int64 
 1   Text       1490 non-null   object
 2   Category   1490 non-null   object
dtypes: int64(1), object(2)
memory usage: 35.1+ KB


In [6]:
from newsbot.preprocessing import clean_text

In [7]:
df['clean_text'] = df['Text'].apply(clean_text)

df['clean_text'].head()

,clean_text
0,worldcom exboss launches defence lawyers defen...
1,german business confidence slides german busin...
2,bbc poll indicates economic gloom citizens in ...
3,lifestyle governs mobile choice faster better ...
4,enron bosses in m payout eighteen former enron...


In [8]:
import spacy
from collections import Counter
nlp = spacy.load('en_core_web_sm')


def extract_syntactic_features(text):

  if not text or pd.isna(text):
    return {}


  doc = nlp(text)

  features = {
        'num_sentences': len(list(doc.sents)),
        'num_tokens': len(doc),
        'dependency_relations': [],
        'noun_phrases': [],
        'verb_phrases': [],
        'subjects': [],
        'objects': []
    }

  dependency_relations = []


  for token in doc:
      if token.is_space or token.is_punct:
          continue

      dependency_relations.append(token.dep_)

      if token.dep_ in ['nsubj', 'nsubjpass']:
          features['subjects'].append(token.text.lower())

      elif token.dep_ in ['dobj', 'iobj', 'pobj']:
          features['objects'].append(token.text.lower())

  for chunk in doc.noun_chunks:
      features['noun_phrases'].append(chunk.text.lower())

  features['dependency_counts'] = dict(Counter(dependency_relations))

  result = f'Sentences: {features['num_sentences']}, Subjects: {features['subjects']}, Objects: {features['objects']}, Noun phrases: {features['noun_phrases']}'
  return result




In [9]:
sample = "Apple bought Beats for three billion dollars last year."
extract_syntactic_features(sample)



"Sentences: 1, Subjects: ['apple'], Objects: ['beats', 'dollars'], Noun phrases: ['apple', 'beats', 'three billion dollars']"

In [10]:
%%writefile /content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM/newsbot/extract_syntactic_features.py

import pandas as pd

import spacy
from collections import Counter
nlp = spacy.load('en_core_web_sm')


def extract_syntactic_features(text):

  if not text or pd.isna(text):
    return {}


  doc = nlp(text)

  features = {
        'num_sentences': len(list(doc.sents)),
        'num_tokens': len(doc),
        'dependency_relations': [],
        'noun_phrases': [],
        'verb_phrases': [],
        'subjects': [],
        'objects': []
    }

  dependency_relations = []


  for token in doc:
      if token.is_space or token.is_punct:
          continue

      dependency_relations.append(token.dep_)

      if token.dep_ in ['nsubj', 'nsubjpass']:
          features['subjects'].append(token.text.lower())

      elif token.dep_ in ['dobj', 'iobj', 'pobj']:
          features['objects'].append(token.text.lower())

  for chunk in doc.noun_chunks:
      features['noun_phrases'].append(chunk.text.lower())

  features['dependency_counts'] = dict(Counter(dependency_relations))

  result = f'Sentences: {features['num_sentences']}, Subjects: {features['subjects']}, Objects: {features['objects']}, Noun phrases: {features['noun_phrases']}'
  return result


Writing /content/drive/MyDrive/ITAI2373-NEWSBT-MIDTERM/newsbot/extract_syntactic_features.py
